# CGFS feature-selection benchmark — weighted vs unweighted

Same experiment protocol, extended with two more selectors, four more
classifiers, per-budget timing breakdowns and a false-positive-rate metric.

**What changed from the previous version**

| | previous | this notebook |
|---|---|---|
| selectors | 9 (+ optional mRMR) | **+ ReliefF, + L1** |
| classifiers | random forest | **random forest, decision tree, gradient boosting (histogram config), SVM, custom k-NN** |
| timing | one selection time per seed | **selection, fit and inference time recorded at every budget** |
| metrics | Acc / Prec / Rec / F1 / macro-F1 | **+ FPR** |
| selection runs | once at `k = 30`, prefixes reused | **once per budget**, so RFE and the greedy are timed at the budget they actually produce |

**Protocol (per seed)**

1. Load and preprocess exactly as `load_dataset` does. Column order untouched.
2. Stratified 70/30 split with that seed.
3. `screen_columns` on the training split, then `zscore` with training statistics.
4. Build `sim` and the MI weights `w` once — this shared prep cost is reported
   separately from selection time.
5. For every selector and every budget `k`, run the selector **at that budget**
   and record how long it took.
6. Fit each classifier on each selected subset, recording fit time and inference
   time on the full test split.

Aggregated as **mean ± std** (ddof=1) over the seeds.

**Selectors**

| Name | What it is |
|---|---|
| `CGFS-weighted` | `cgfs(sim, w, k)` with `w_i = d·MI(f_i;y)/Σ MI` |
| `CGFS-unweighted` | `cgfs(sim, 1, k)` — the same greedy, uniform weights |
| `Random` | random subset, the baseline everything must beat |
| `RFE` | recursive feature elimination, random-forest estimator |
| `XGBoost` | gain-based importance |
| `MutualInfo` | top-k by `mutual_info_classif` (kNN estimator), computed inline so its cost is its own — the control that decides whether the coverage machinery earns its place |
| `ReliefF` | Kononenko's multiclass Relief, written out in §10 |
| `L1` | entry order along an L1-logistic regularisation path |
| `Correlation` | max over classes of \|Pearson r\| against the class indicator |
| `ANOVA` | `f_classif` |
| `Chi2` | `chi2` on min–max scaled features |
| `mRMR` | optional (`INCLUDE_MRMR`), the script's MID variant |

**Classifiers** — `rf`, `dt`, `gb`, `svm`, `knn` by default; `logreg` also defined.
`gb` is gradient boosting in its fast histogram configuration (XGBoost `hist`
when importable, otherwise `HistGradientBoostingClassifier`), never the exact
`GradientBoostingClassifier`. `knn` is a hand-written brute-force k-NN.

**Runtime.** This grid is large: `selectors × budgets × classifiers × seeds`
model fits, plus a selection run per budget. Read §12's estimate before starting
a full run. The knobs that cut it fastest are `CLASSIFIERS_TO_RUN`,
`SELECT_PER_K = False`, `KNN_MAX_TRAIN` and `RFE_SUBSAMPLE`.

`faiss` and `xgboost` are optional — the notebook falls back to exact
NumPy/BLAS and to random-forest gain respectively, and says so.

## 1. Configuration

In [1]:
# ---------------------------------------------------------------- config ----
DATASET_KEY = "rt-iot"           # "rt-iot" or "edge-iiot" (specs in section 3)
DATA_DIR    = "."                # directory holding the csv
OUT_DIR     = "."                # where result csv / tex files are written

SEEDS   = [0, 1, 2 ]#[3, 5]
K_LIST  = [5, 10, 15, 20, 25, 30]
TEST_SIZE = 0.30
N_JOBS  = -1

# --- the script's knobs, same defaults ---------------------------------------
MI_BINS           = 32
NEAR_CONST_THRESH = 0.995        # drop a column whose modal value covers >= this
SCREEN            = True         # set False to exercise Proposition 3 directly
DROP_LEAKY        = True         # drop host/timestamp/payload columns

# Column order is left exactly as it comes out of the csv. Each seed is an
# independent stratified split and also seeds the random baseline and every
# estimator, so the spread reported below is resampling variance.

# Which MI the `MutualInfo` baseline ranks by.
#   "sklearn" -> mutual_info_classif, the Kraskov-style kNN estimator. Computed
#                inside the selector call, so select_time_s is genuinely its own
#                cost and no shared prep is charged to it. This is a different
#                estimator from the one behind the CGFS weights, which makes it an
#                independent baseline rather than a re-ranking of the same numbers.
#   "plugin"  -> the script's own binned plug-in MI, i.e. the same numbers that
#                define the CGFS weights. Tighter as a control for the weighting
#                specifically, but already computed in build_context, so its cost
#                shows up as charged prep rather than as measured selection time.
MI_BASELINE = "sklearn"
MI_N_NEIGHBORS = 3               # mutual_info_classif's kNN parameter

INCLUDE_MRMR = True             # add the script's mRMR (MID) as an extra selector

# --- ReliefF ------------------------------------------------------------------
RELIEFF_K         = 10           # neighbours per class
RELIEFF_N_SAMPLES = 1_000        # instances sampled to accumulate the weights
RELIEFF_POOL      = 5_000        # stratified reference pool the neighbours come from
RELIEFF_CHUNK     = 256          # query rows per distance block

# --- L1 -----------------------------------------------------------------------
L1_C_GRID    = [0.005, 0.02, 0.1, 0.5, 2.0, 10.0]   # strong -> weak regularisation
L1_SOLVER    = "saga"            # the only solver that does multinomial L1; liblinear
                                 # is binary-only in current scikit-learn
L1_MAX_ITER  = 300
L1_TOL       = 1e-3              # loose: the ranking only needs the support, not a
                                 # converged optimum
L1_SUBSAMPLE = 20_000

# --- gradient boosting: the fast histogram configuration ----------------------
GB_BACKEND      = "auto"         # "auto" -> xgboost hist if importable, else sklearn HGB
GB_N_ESTIMATORS = 100
GB_MAX_DEPTH    = 6
GB_LR           = 0.3
GB_MAX_BINS     = 128            # fewer bins = faster; sklearn HGB caps this at 255

# --- SVM ----------------------------------------------------------------------
SVM_KIND     = "linear"          # "linear" (LinearSVC), "sgd" (hinge-loss SGD), "rbf" (SVC)
SVM_MAX_ITER = 2_000
SVM_C        = 1.0
SVM_TOL      = 1e-3              # liblinear's default is 1e-4; the extra digit costs a
                                 # lot of iterations and moves the boundary very little
SVM_PARALLEL_OVR = True          # fit the one-vs-rest problems across cores. Same model,
                                 # same numbers -- liblinear's own multiclass loop is
                                 # one-vs-rest too, but single-threaded.
SVM_SGD_ALPHA    = 1e-5          # only for SVM_KIND == "sgd". Same hinge + L2 objective
                                 # as LinearSVC, with alpha ~ 1/(C*n_train).
SVM_SGD_MAX_ITER = 50
SVM_RBF_MAX_TRAIN = 20_000       # only used when SVM_KIND == "rbf"

# --- custom k-NN --------------------------------------------------------------
KNN_NEIGHBORS = 5
KNN_WEIGHTS   = "distance"       # "distance" or "uniform"
KNN_CHUNK     = 512              # query rows per distance block
KNN_MAX_TRAIN = 20_000           # reference-set cap; None uses the whole training split

# If True  : every selector is re-run at each budget, so select_time_s is the
#            real cost of producing that many features (RFE in particular is far
#            more expensive at small k).
# If False : one run at max(K_LIST), prefixes reused, and the single measured
#            time is repeated across budgets. Much faster, less honest timing.
SELECT_PER_K = True

# The benign/normal class, or a list of them -- RT-IoT2022 has several
# (e.g. ["MQTT_Publish", "Thing_Speak", "Wipro_bulb"]), Edge-IIoTset has one
# ("Normal"). When set, a binary false-alarm rate is reported alongside the macro
# FPR: the fraction of true-benign flows predicted as some attack. A benign flow
# predicted as a *different* benign class is not a false alarm and is not counted.
# Leave as None to skip it.
NORMAL_CLASS = ["MQTT_Publish","Thing_Speak","Wipro_bulb"]#RT-IoT2022 Normal classes#["Normal"] #Edge-IIoT #["BenignTraffic"] #CIC-IoT2023
#["MQTT_Publish","Thing_Speak","Wipro_bulb"]#RT-IoT2022 Normal classes #None

CLASSIFIERS_TO_RUN = ["rf", "dt", "gb", "svm", "knn"]   # "logreg" also available

# Subsample sizes for the two expensive selectors (stratified, train split only).
RFE_SUBSAMPLE = None#20_000
MI_SUBSAMPLE  = None#20_000

RFE_STEP         = 1             # raise to 2-3, or lower RFE_SUBSAMPLE, to speed RFE up
RFE_N_ESTIMATORS = 50
CLF_N_ESTIMATORS = 100           # downstream random forest, as in the script
XGB_N_ESTIMATORS = 200

RUN_SELF_CHECK = True
VERBOSE = True

## 2. Imports and optional dependencies

In [2]:
import heapq, inspect, time, warnings, os
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE, mutual_info_classif, f_classif, chi2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

try:
    import faiss
    HAVE_FAISS = True
except ImportError:
    HAVE_FAISS = False

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False

print(f"numpy {np.__version__} | pandas {pd.__version__}")
print(f"faiss   : {'yes' if HAVE_FAISS else 'NO  -> exact NumPy/BLAS fallback (identical similarities)'}")
print(f"xgboost : {'yes' if HAVE_XGB else 'NO  -> XGBoost selector falls back to random-forest gini importance'}")

numpy 1.26.4 | pandas 2.1.4
faiss   : yes
xgboost : yes


## 3. Dataset specs and loading

`DatasetSpec` and `load_dataset` verbatim from the script. Leaky columns are
those that identify hosts/sessions or carry raw payload and timestamps — left
in, they let the classifier memorise the capture rather than learn the attack,
and they push the full-feature score to a perfect 1.0000 on Edge-IIoTset.
RT-IoT2022 has none declared.

The one addition is an `inf → NaN` step so the existing `fillna(0)` catches it;
a single infinite value otherwise kills every scikit-learn estimator downstream.

In [3]:
@dataclass(frozen=True)
class DatasetSpec:
    name: str
    path: str
    target: str
    drop: list
    # Columns that identify hosts/sessions or are raw payload/timestamps.
    leaky: list = field(default_factory=list)


DATASETS = {
    "rt-iot": DatasetSpec(
        name="RT-IoT2022",
        path="../RT_IOT2022.csv",
        target="Attack_type",
        drop=["Unnamed: 0", "Attack_type", "Attack_Label"],
        leaky=[],
    ),
    "edge-iiot": DatasetSpec(
        name="Edge-IIoTset",
        path="../ML-EdgeIIoT-dataset.csv",
        target="Attack_type",
        drop=["Attack_label", "Attack_type"],
        leaky=[
            "frame.time", "ip.src_host", "ip.dst_host",
            "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
            "http.request.full_uri", "http.file_data", "http.request.uri.query",
            "tcp.payload", "tcp.options", "mqtt.msg", "dns.qry.name",
        ],
    ),
        "cic-iot": DatasetSpec(
        name="CICIoT2023",
        path="../CICIoT2023.csv",
        target="label",
        drop=['label', 'Binary Label'],
        leaky=[
        ],
    ),
}


def load_dataset(spec: DatasetSpec, drop_leaky: bool):
    """Read the CSV, drop target/leaky columns, integer-encode the rest."""
    path = os.path.join(DATA_DIR, spec.path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Set DATA_DIR (and DATASET_KEY) in the config cell.")
    df = pd.read_csv(path, low_memory=False)
    # for CIC-IoT2023 dataset only
    # sample_fraction = 1/7
    # df = df.groupby('label', group_keys=False).apply(lambda x: x.sample(frac=sample_fraction,random_state=42))
    
    y = df[spec.target].values
    X = df.drop([c for c in spec.drop if c in df.columns], axis=1)
    if drop_leaky:
        dropped_leaky = [c for c in spec.leaky if c in X.columns]
        X = X.drop(dropped_leaky, axis=1)
    else:
        dropped_leaky = []

    # Test numeric-ness rather than `dtype == 'object'`: on pandas >= 3.0 text
    # columns carry the dedicated 'str' dtype, so the older check silently skips
    # them and the string values reach the similarity computation unencoded.
    encoded = []
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))
            encoded.append(col)

    X = X.replace([np.inf, -np.inf], np.nan)
    n_filled = int(X.isna().to_numpy().sum())
    X = X.fillna(0)

    print(f"{spec.name}: {X.shape[0]:,} rows x {X.shape[1]} columns "
          f"({'leaky columns dropped: ' + str(dropped_leaky) if drop_leaky else 'LEAKY COLUMNS KEPT'})")
    print(f"  label-encoded columns : {encoded if encoded else 'none'}")
    print(f"  non-finite/missing cells filled with 0 : {n_filled:,}")
    return X.values.astype(np.float64), y, list(X.columns)

## 4. Screening and standardisation

Both verbatim from the script. Screening runs on the **training split only** and
removes constant and near-constant columns; Proposition 3 (orphan exclusion)
says the weighted objective would never select these anyway, so screening just
keeps them out of the similarity matrix. Set `SCREEN = False` to leave them in
and exercise that proposition directly against the unweighted baseline.

In [4]:
def screen_columns(Xtr: np.ndarray, cols: list, enabled: bool,
                   near_const_thresh):
    """Indices of columns kept after constant / near-constant screening."""
    if not enabled:
        return list(range(Xtr.shape[1])), list(cols)
    keep = []
    n = len(Xtr)
    for j in range(Xtr.shape[1]):
        col = Xtr[:, j]
        if len(np.unique(col)) <= 1:
            continue
        if near_const_thresh is not None:
            _, counts = np.unique(col, return_counts=True)
            if counts.max() / n >= near_const_thresh:
                continue
        keep.append(j)
    return keep, [cols[j] for j in keep]


def zscore(Xtr: np.ndarray, Xte: np.ndarray):
    """Standardize with training statistics only."""
    mu = Xtr.mean(axis=0)
    sd = Xtr.std(axis=0)
    sd[sd == 0] = 1.0
    return (Xtr - mu) / sd, (Xte - mu) / sd

## 5. Rectified cosine similarity and MI relevance weights

From the script. The similarity build keeps the flat-index requirement — the
guarantees assume an exact oracle for `G`, and the complete row is retrieved
(`m = d`) rather than a top-`m` list, because a top-`m` inner-product query
returns only the most positively aligned features and would drop the
anti-correlated ones that rectification makes maximally similar.

`_exact_ip_search` is the only addition: it reproduces
`IndexFlatIP.search` exactly in NumPy so the notebook runs without `faiss`.

In [5]:
def _l2_normalize_rows(M):
    """faiss.normalize_L2 semantics: zero rows are left as zero rows."""
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return M / norms


def _exact_ip_search(feat, m):
    """Exact drop-in for faiss IndexFlatIP(...).search(feat, m)."""
    S = feat @ feat.T
    I = np.argsort(-S, axis=1, kind="stable")[:, :m]
    D = np.take_along_axis(S, I, axis=1)
    return D.astype(np.float32), I.astype(np.int64)


def rectified_cosine_matrix(Z: np.ndarray) -> np.ndarray:
    """s(f_i,f_j) = |cos(f_i,f_j)| for every pair, via an exact flat index."""
    d = Z.shape[1]
    feat = Z.T.astype(np.float32).copy()           # (d, n): one row per feature
    if HAVE_FAISS:
        faiss.normalize_L2(feat)
        index = faiss.IndexFlatIP(feat.shape[1])
        index.add(feat)
        D, I = index.search(feat, d)
    else:
        feat = _l2_normalize_rows(feat)
        D, I = _exact_ip_search(feat, d)
    sim = np.empty((d, d), dtype=np.float64)
    np.put_along_axis(sim, I, D.astype(np.float64), axis=1)   # back to column order
    sim = np.abs(sim)
    np.fill_diagonal(sim, 1.0)                     # s(f_i,f_i) = 1 exactly
    return sim


def _bin_column(col: np.ndarray, bins: int) -> np.ndarray:
    """Quantile-bin a column; low-cardinality columns are used as-is (exact)."""
    uniq = np.unique(col)
    if len(uniq) <= bins:
        return np.searchsorted(uniq, col)
    edges = np.quantile(col, np.linspace(0.0, 1.0, bins + 1)[1:-1])
    return np.digitize(col, np.unique(edges))


def _mi_from_codes(a: np.ndarray, b: np.ndarray) -> float:
    """Plug-in mutual information between two integer-coded variables, in nats."""
    joint = np.histogram2d(a, b, bins=(a.max() + 1, b.max() + 1))[0]
    P = joint / joint.sum()
    px = P.sum(axis=1, keepdims=True)
    py = P.sum(axis=0, keepdims=True)
    nz = P > 0
    return float((P[nz] * np.log(P[nz] / (px @ py)[nz])).sum())


def mi_weights(Xtr: np.ndarray, ytr: np.ndarray, bins: int, weighted: bool = True):
    """Relevance weights w_i, normalized so that sum_i w_i = d.

    Returns (w, raw_mi, binned_columns). With weighted=False returns w == 1,
    which recovers the unweighted formulation while keeping sum_i w_i = d.
    """
    d = Xtr.shape[1]
    codes = [_bin_column(Xtr[:, j], bins) for j in range(d)]
    ycodes = pd.factorize(ytr)[0]
    mi = np.array([_mi_from_codes(c, ycodes) for c in codes])
    if not weighted:
        return np.ones(d), mi, codes
    total = mi.sum()
    # Degenerate case: no feature carries any label information.
    w = np.full(d, 1.0) if total <= 0 else d * mi / total
    return w, mi, codes

## 6. The CGFS greedy, and the script's MI / mRMR selectors

`cgfs` verbatim. One implementation serves both variants: pass the MI weights
for `CGFS-weighted`, pass `np.ones(d)` for `CGFS-unweighted`. `cgfs_naive` is
kept because the self-check in §7 uses it to prove the lazy version is exact.

In [6]:
def cgfs(sim: np.ndarray, w: np.ndarray, k: int):
    """Algorithm 1 (CGFS) via lazy greedy.

    Gain is  G(e | F_s) = sum_i w_i * max{0, s(f_i,f_e) - g_i}, and the recorded
    trace is E_i = d - sum_i w_i g_i. Stale priority-queue keys are upper bounds
    on current gains (submodularity), so the extracted feature is always the true
    maximizer and the output matches the standard greedy exactly.
    """
    d = sim.shape[0]
    k = min(k, d)
    g = np.zeros(d)
    selected = []
    errors = []

    def gain(e: int) -> float:
        return float((w * np.maximum(0.0, sim[:, e] - g)).sum())

    heap = [(-gain(e), e) for e in range(d)]
    heapq.heapify(heap)

    while len(selected) < k and heap:
        _, e = heapq.heappop(heap)
        refreshed = gain(e)
        if not heap or refreshed >= -heap[0][0] - 1e-12:
            np.maximum(g, sim[:, e], out=g)
            selected.append(e)
            errors.append(float(d - (w * g).sum()))
        else:
            heapq.heappush(heap, (-refreshed, e))
    return selected, errors


def cgfs_naive(sim: np.ndarray, w: np.ndarray, k: int):
    """Standard (non-lazy) greedy, used only to verify the lazy implementation."""
    d = sim.shape[0]
    g = np.zeros(d)
    selected = []
    for _ in range(min(k, d)):
        gains = np.array([
            -np.inf if e in selected else float((w * np.maximum(0.0, sim[:, e] - g)).sum())
            for e in range(d)
        ])
        e = int(gains.argmax())
        np.maximum(g, sim[:, e], out=g)
        selected.append(e)
    return selected


def mi_ranking(mi: np.ndarray, k: int):
    """Top-k by mutual information, ignoring redundancy entirely.

    The critical control: if this matches weighted CGFS, the submodular coverage
    machinery is not earning its place.
    """
    return list(np.argsort(-mi)[:k]), []


def mrmr(mi: np.ndarray, codes: list, k: int):
    """Max-relevance min-redundancy (MID): argmax [ MI(f;y) - mean_{s in S} MI(f;s) ]."""
    d = len(mi)
    k = min(k, d)
    selected = [int(np.argmax(mi))]
    redundancy = np.zeros(d)
    ff_cache = {}

    def ff_mi(a: int, b: int) -> float:
        key = (a, b) if a < b else (b, a)
        if key not in ff_cache:
            ff_cache[key] = _mi_from_codes(codes[key[0]], codes[key[1]])
        return ff_cache[key]

    while len(selected) < k:
        last = selected[-1]
        best, best_score = -1, -np.inf
        for e in range(d):
            if e in selected:
                continue
            redundancy[e] += ff_mi(e, last)
            score = mi[e] - redundancy[e] / len(selected)
            if score > best_score:
                best, best_score = e, float(score)
        selected.append(best)
    return selected, []

## 7. Self-check

The script's `self_check()`, run before the experiment: similarity symmetry and
range, `Σ w = d`, non-negative weights, **lazy greedy == standard greedy** for
both variants, submodularity on 200 random pairs, monotone coverage error, and
Proposition 3 (the constant column is never selected and carries zero weight).

In [7]:
def self_check() -> int:
    print("running self-checks...")
    rng = np.random.default_rng(0)
    failures = 0

    # A matrix with duplicated blocks (redundancy), noise, and a constant column.
    base = rng.normal(size=(400, 4))
    X = np.hstack([base, base + 0.01 * rng.normal(size=(400, 4)),
                   rng.normal(size=(400, 3)), np.full((400, 1), 7.0)])
    y = (base[:, 0] + 0.3 * rng.normal(size=400) > 0).astype(int)

    Ztr, _ = zscore(X, X)
    sim = rectified_cosine_matrix(Ztr)
    d = sim.shape[1]

    if not np.allclose(sim, sim.T, atol=1e-5):
        print("  FAIL: similarity matrix is not symmetric"); failures += 1
    else:
        print("  ok: similarity symmetric")
    if not (sim.min() >= -1e-9 and sim.max() <= 1 + 1e-6):
        print("  FAIL: rectified similarity outside [0,1]"); failures += 1
    else:
        print("  ok: similarity in [0,1]")

    w, mi, codes = mi_weights(X, y, bins=32, weighted=True)
    if not np.isclose(w.sum(), d):
        print(f"  FAIL: weights sum to {w.sum():.4f}, expected d={d}"); failures += 1
    else:
        print(f"  ok: sum(w) = d = {d}")
    if not (w >= 0).all():
        print("  FAIL: negative weight"); failures += 1
    else:
        print("  ok: weights non-negative")

    # Lazy greedy must reproduce the standard greedy exactly (both variants).
    for tag, wv in (("weighted", w), ("unweighted", np.ones(d))):
        lazy, _ = cgfs(sim, wv, 8)
        naive = cgfs_naive(sim, wv, 8)
        if lazy != naive:
            print(f"  FAIL: lazy != naive greedy ({tag}): {lazy} vs {naive}"); failures += 1
        else:
            print(f"  ok: lazy greedy == standard greedy ({tag})")

    # Diminishing returns: G(e|S) >= G(e|T) for S subset of T.
    def G(S, wv):
        if not S:
            return 0.0
        return float((wv * sim[:, list(S)].max(axis=1)).sum())

    worst = 0.0
    for _ in range(200):
        T = set(rng.choice(d, rng.integers(2, 6), replace=False).tolist())
        S = set(list(T)[: max(1, len(T) - 2)])
        rest = [e for e in range(d) if e not in T]
        if not rest:
            continue
        e = int(rng.choice(rest))
        worst = min(worst, (G(S | {e}, w) - G(S, w)) - (G(T | {e}, w) - G(T, w)))
    if worst < -1e-9:
        print(f"  FAIL: submodularity violated by {worst:.2e}"); failures += 1
    else:
        print("  ok: submodularity holds on 200 random pairs")

    # Coverage error must be non-increasing.
    _, errs = cgfs(sim, w, 10)
    if any(b > a + 1e-9 for a, b in zip(errs, errs[1:])):
        print("  FAIL: coverage error not monotone"); failures += 1
    else:
        print("  ok: coverage error non-increasing")

    # Proposition 3: the constant column (last, index d-1) is never selected.
    sel, _ = cgfs(sim, w, d - 1)
    if (d - 1) in sel[: d - 2]:
        print("  FAIL: constant column selected by weighted CGFS"); failures += 1
    else:
        print("  ok: constant column excluded by weighted CGFS")
    if w[d - 1] > 1e-9:
        print(f"  FAIL: constant column has weight {w[d-1]:.3e}"); failures += 1
    else:
        print("  ok: constant column has zero MI weight")

    print("self-checks passed" if not failures else f"{failures} self-check FAILURE(S)")
    return failures


if RUN_SELF_CHECK:
    assert self_check() == 0, "self-checks failed -- fix before trusting the results"

running self-checks...
  ok: similarity symmetric
  ok: similarity in [0,1]
  ok: sum(w) = d = 12
  ok: weights non-negative
  ok: lazy greedy == standard greedy (weighted)
  ok: lazy greedy == standard greedy (unweighted)
  ok: submodularity holds on 200 random pairs
  ok: coverage error non-increasing
  ok: constant column excluded by weighted CGFS
  ok: constant column has zero MI weight
self-checks passed


## 8. Load the dataset

In [8]:
SPEC = DATASETS[DATASET_KEY]
X_ALL, Y_RAW, COLS = load_dataset(SPEC, drop_leaky=DROP_LEAKY)

LE_Y = LabelEncoder()
Y_ALL = LE_Y.fit_transform(Y_RAW)
D_TOTAL = X_ALL.shape[1]

counts = pd.Series(Y_RAW).value_counts()
print(f"  {len(LE_Y.classes_)} classes | support min={counts.min():,} ({counts.idxmin()}), "
      f"max={counts.max():,} ({counts.idxmax()})")
print(f"\nbudgets requested: {K_LIST} of {D_TOTAL} pre-screening columns")

RT-IoT2022: 123,117 rows x 83 columns (leaky columns dropped: [])
  label-encoded columns : ['proto', 'service']
  non-finite/missing cells filled with 0 : 0
  12 classes | support min=28 (NMAP_FIN_SCAN), max=94,659 (DOS_SYN_Hping)

budgets requested: [5, 10, 15, 20, 25, 30] of 83 pre-screening columns


## 9. Per-seed context

The script's per-seed pipeline unchanged: split, screen, standardise. `sim`,
`w`, `mi` and `codes` are built once here and shared by both CGFS variants — the
same shared state the script uses, so the weighted/unweighted comparison differs
in nothing but `w`.

Screening runs after the split, so the surviving feature count `d` can differ
slightly between seeds; the run log prints it per seed.

In [9]:
def stratified_subsample(y, n, rng):
    """Proportional stratified subsample keeping at least one row per class."""
    if n is None or n >= len(y):
        return np.arange(len(y))
    idx = []
    classes, counts = np.unique(y, return_counts=True)
    for c, cnt in zip(classes, counts):
        pos = np.where(y == c)[0]
        take = min(cnt, max(1, int(round(cnt * n / len(y)))))
        idx.append(rng.choice(pos, size=take, replace=False))
    return np.sort(np.concatenate(idx))


@dataclass
class SeedContext:
    seed: int                # drives the split, the random baseline and every estimator
    keep: list               # indices of the columns surviving screening
    feat_names: list         # names of the kept features, in kept order
    d: int
    Xtr: np.ndarray          # raw, screened
    Xte: np.ndarray
    Ztr: np.ndarray          # z-scored with train stats, screened
    Zte: np.ndarray
    Xtr_mm: np.ndarray       # min-max of Xtr in [0, 1], for chi2
    ytr: np.ndarray
    yte: np.ndarray
    sim: np.ndarray          # rectified cosine matrix on Ztr
    w: np.ndarray            # MI relevance weights, sum = d
    mi: np.ndarray           # raw plug-in MI per feature
    codes: list              # binned columns, for mRMR
    prep_s: float            # total shared prep = prep_sim_s + prep_mi_s
    prep_sim_s: float        # seconds to build the rectified cosine matrix
    prep_mi_s: float         # seconds to bin the columns and score the plug-in MI
    sub_rfe: np.ndarray
    sub_mi: np.ndarray
    sub_l1: np.ndarray


def build_context(seed):
    sub_rng = np.random.default_rng(seed)

    # Column order is left as it comes out of the csv; the seed picks the split.
    Xtr_raw, Xte_raw, ytr, yte = train_test_split(
        X_ALL, Y_ALL, test_size=TEST_SIZE, stratify=Y_ALL, random_state=seed)

    keep, kept_cols = screen_columns(Xtr_raw, COLS, SCREEN, NEAR_CONST_THRESH)
    Xtr, Xte = Xtr_raw[:, keep], Xte_raw[:, keep]
    Ztr, Zte = zscore(Xtr, Xte)
    d = Ztr.shape[1]

    # Timed separately so the cost can be charged back to whichever selectors
    # actually consume it (see SELECTOR_PREP at the end of section 10).
    t0 = time.perf_counter()
    sim = rectified_cosine_matrix(Ztr)
    prep_sim_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    w, mi, codes = mi_weights(Xtr, ytr, MI_BINS, weighted=True)
    prep_mi_s = time.perf_counter() - t0

    lo, hi = Xtr.min(axis=0), Xtr.max(axis=0)
    span = hi - lo
    span[span == 0] = 1.0
    Xtr_mm = np.clip((Xtr - lo) / span, 0.0, None)

    return SeedContext(
        seed=seed, keep=keep, feat_names=kept_cols, d=d,
        Xtr=Xtr, Xte=Xte, Ztr=Ztr, Zte=Zte, Xtr_mm=Xtr_mm, ytr=ytr, yte=yte,
        sim=sim, w=w, mi=mi, codes=codes,
        prep_s=prep_sim_s + prep_mi_s, prep_sim_s=prep_sim_s, prep_mi_s=prep_mi_s,
        sub_rfe=stratified_subsample(ytr, RFE_SUBSAMPLE, sub_rng),
        sub_mi=stratified_subsample(ytr, MI_SUBSAMPLE, sub_rng),
        sub_l1=stratified_subsample(ytr, L1_SUBSAMPLE, sub_rng),
    )

## 10. The selectors

Each selector takes a budget `k` and returns exactly `k` feature indices in the
screened space. With `SELECT_PER_K = True` the run loop calls every selector once
per budget and times that call, so `select_time_s` is the true cost of producing
that many features — which matters most for RFE, whose work grows as `k` shrinks.

Score-based selectors recompute their whole score vector for each budget, so
their time comes out roughly constant in `k`; that is the honest number, not an
artefact.

**Shared prep is charged back.** Three selectors read precomputed state out of
`build_context` rather than computing it inside their own call, which would make
them look free next to selectors that do all their work inline:

| selector | reads | so `select_time_s` alone omits |
|---|---|---|
| `CGFS-weighted` | `sim`, `w` | the similarity matrix **and** the MI |
| `CGFS-unweighted` | `sim` | the similarity matrix |
| `MutualInfo` (only when `MI_BASELINE = "plugin"`) | `mi` | the MI — its own entire score, hence a measured time of ~0 |
| `mRMR` | `mi`, `codes` | the MI and the binning |

`SELECTOR_PREP` at the end of this section declares those dependencies, and the
run loop adds the matching prep to give **`select_total_s`**. That is the column
to quote: it is what each method costs from raw training matrix to chosen subset,
on the same footing as `ReliefF` or `RFE`, which do everything inline and so have
`select_total_s == select_time_s`.

One caveat: the prep is built once per seed, so if you run all six budgets it is
paid once and charged to each — right for a per-budget deployment cost, pessimistic
if you read the table as a per-seed total.

At the default `MI_BASELINE = "sklearn"` the `MutualInfo` row is *not* in that
table. It calls `mutual_info_classif` inside its own selector call, so its time is
measured the same way `ReliefF`'s is and nothing is charged to it. Note that this
estimator is a nearest-neighbour one, not the binned plug-in used for the CGFS
weights, so the baseline is genuinely independent rather than a re-ranking of the
weight vector — and it is much slower, since it does a kNN search per feature. It
is also k-independent, so `SELECT_PER_K = True` recomputes the whole score vector
at every budget.

Ties are broken by a *stable* argsort, i.e. by column position in the csv, so a
tie resolves the same way in every seed.

**ReliefF** and **L1** are written out below.

* `relieff_scores` is Kononenko's multiclass Relief: for each sampled instance,
  push a feature's weight down by its distance to the `k` nearest same-class
  neighbours (hits) and up by its distance to the `k` nearest neighbours of each
  other class (misses), the miss terms weighted by the class prior renormalised
  over the other classes. Neighbours are found under Manhattan distance on
  range-normalised features, the standard choice for continuous attributes.
  Unlike every filter above it, this scores a feature by how well it separates
  neighbouring points, so it sees interactions the univariate filters cannot.
* `l1_path_order` sweeps the inverse-regularisation strength `C` from strong to
  weak and records the order in which coefficients leave zero. A single L1 fit
  cannot rank a top-k beyond its own support — everything it zeroes out is tied
  — so the path is what makes an L1 ranking well defined at every budget. The
  solver is `saga`: `liblinear` is binary-only in current scikit-learn, and
  wrapping it one-vs-rest would rank against a different objective than the
  multinomial one being fitted.

In [10]:
def order_from_scores(scores, k=None):
    """Descending order, NaNs last, ties broken by column position."""
    s = np.asarray(scores, dtype=np.float64)
    s = np.where(np.isfinite(s), s, -np.inf)
    order = list(np.argsort(-s, kind="stable"))
    return order if k is None else order[:k]


# --------------------------------------------------------------------- ReliefF
def relieff_scores(X, y, n_samples, n_pool, k_neighbors, rng, chunk):
    """ReliefF (Kononenko, 1994).

        W[A] = sum over sampled R of
                 - sum_{H in kNN hits}  diff(A, R, H) / (m*k)
                 + sum_{C != class(R)} [P(C) / (1 - P(class(R)))]
                       * sum_{M in kNN misses(C)} diff(A, R, M) / (m*k)

    diff is the range-normalised absolute difference; neighbours are found under
    Manhattan distance on the same normalisation.
    """
    n, d = X.shape
    pool = stratified_subsample(y, min(n_pool, n), rng)
    P, yp = X[pool], y[pool]
    lo, hi = P.min(axis=0), P.max(axis=0)
    span = hi - lo
    span[span == 0] = 1.0
    Pn = ((P - lo) / span).astype(np.float32)

    classes, counts = np.unique(yp, return_counts=True)
    prior = dict(zip(classes.tolist(), (counts / len(yp)).tolist()))
    rows_of = {int(c): np.where(yp == c)[0] for c in classes}

    m = int(min(n_samples, len(Pn)))
    samp = rng.choice(len(Pn), size=m, replace=False)

    W = np.zeros(d)
    for start in range(0, m, chunk):
        block = samp[start:start + chunk]
        Q = Pn[block]
        # Manhattan distance accumulated one feature at a time. The obvious
        # (b, n_pool, d) broadcast is the same arithmetic but would need tens of
        # gigabytes at these pool sizes.
        Dm = np.zeros((len(block), len(Pn)), dtype=np.float32)
        for j in range(d):
            Dm += np.abs(Q[:, j:j + 1] - Pn[None, :, j])

        for bi, gi in enumerate(block):
            c = int(yp[gi])
            drow = Dm[bi]

            rows = rows_of[c]
            kh = min(k_neighbors, len(rows) - 1)
            if kh > 0:
                cand = rows[np.argpartition(drow[rows], kh)[:kh + 1]]
                cand = cand[cand != gi][:kh]
                W -= np.abs(Pn[gi] - Pn[cand]).sum(axis=0) / (m * kh)

            denom = 1.0 - prior[c]
            if denom <= 1e-12:
                continue
            for c2 in classes:
                c2 = int(c2)
                if c2 == c:
                    continue
                rows2 = rows_of[c2]
                km = min(k_neighbors, len(rows2))
                if km == 0:
                    continue
                cand2 = rows2[np.argpartition(drow[rows2], km - 1)[:km]]
                W += (prior[c2] / denom) * np.abs(Pn[gi] - Pn[cand2]).sum(axis=0) / (m * km)
    return W


# -------------------------------------------------------------------------- L1
def l1_path_order(Z, yv, k, seed, C_grid, max_iter):
    """Rank features by where they enter the L1-logistic regularisation path.

    Sweeping C from strong to weak and recording the order in which coefficients
    become non-zero gives a ranking that is defined at every budget; a single fit
    leaves its whole zeroed-out complement tied.
    """
    order, seen, last = [], set(), np.zeros(Z.shape[1])
    for C in C_grid:
        clf = LogisticRegression(penalty="l1", solver=L1_SOLVER, C=C,
                                 max_iter=max_iter, tol=L1_TOL, random_state=seed)
        clf.fit(Z, yv)
        score = np.abs(np.atleast_2d(clf.coef_)).max(axis=0)   # max |coef| over classes
        last = score
        for j in np.argsort(-score, kind="stable"):
            j = int(j)
            if score[j] > 0 and j not in seen:
                seen.add(j)
                order.append(j)
        if len(order) >= k:
            break
    if len(order) < k:            # path never reached k non-zeros: fall back to |coef|
        for j in np.argsort(-last, kind="stable"):
            j = int(j)
            if j not in seen:
                seen.add(j)
                order.append(j)
    return order[:k]


# ------------------------------------------------------------- the selectors
def sel_cgfs_weighted(ctx, k):
    return cgfs(ctx.sim, ctx.w, k)[0]


def sel_cgfs_unweighted(ctx, k):
    return cgfs(ctx.sim, np.ones(ctx.d), k)[0]


def sel_random(ctx, k):
    # A fresh generator, so the baseline depends on the seed but not on how many
    # draws the rest of the pipeline happened to consume. A permutation prefix
    # keeps the budgets nested; the script drew each budget independently.
    return list(np.random.default_rng(10_000 + ctx.seed).permutation(ctx.d))[:k]


def sel_rfe(ctx, k):
    est = RandomForestClassifier(n_estimators=RFE_N_ESTIMATORS, n_jobs=N_JOBS,
                                 random_state=ctx.seed)
    rfe = RFE(estimator=est, n_features_to_select=k, step=RFE_STEP)
    rfe.fit(ctx.Ztr[ctx.sub_rfe], ctx.ytr[ctx.sub_rfe])
    return list(np.argsort(rfe.ranking_, kind="stable"))[:k]


def sel_xgboost(ctx, k):
    if HAVE_XGB:
        clf = XGBClassifier(n_estimators=XGB_N_ESTIMATORS, max_depth=6,
                            learning_rate=0.3, tree_method="hist",
                            importance_type="gain", n_jobs=N_JOBS,
                            random_state=ctx.seed, verbosity=0)
    else:
        clf = RandomForestClassifier(n_estimators=CLF_N_ESTIMATORS, n_jobs=N_JOBS,
                                     random_state=ctx.seed)
    clf.fit(ctx.Ztr, ctx.ytr)
    return order_from_scores(clf.feature_importances_, k)


# mutual_info_classif gained n_jobs in scikit-learn 1.5; pass it only if this
# install accepts it, since the whole point here is to not be slower than needed.
_MI_SUPPORTS_NJOBS = "n_jobs" in inspect.signature(mutual_info_classif).parameters


def sel_mutual_info(ctx, k):
    if MI_BASELINE == "plugin":
        # Same MI that defines the CGFS weights -> the exact control for weighting.
        # Already computed in build_context, so its cost is charged as prep.
        return order_from_scores(ctx.mi, k)
    kwargs = dict(n_neighbors=MI_N_NEIGHBORS, random_state=ctx.seed)
    if _MI_SUPPORTS_NJOBS:
        kwargs["n_jobs"] = N_JOBS
    # Computed here rather than in build_context so the timer sees the real cost.
    mi = mutual_info_classif(ctx.Ztr[ctx.sub_mi], ctx.ytr[ctx.sub_mi], **kwargs)
    return order_from_scores(mi, k)


def sel_relieff(ctx, k):
    rng = np.random.default_rng(20_000 + ctx.seed)
    w = relieff_scores(ctx.Ztr, ctx.ytr, RELIEFF_N_SAMPLES, RELIEFF_POOL,
                       RELIEFF_K, rng, RELIEFF_CHUNK)
    return order_from_scores(w, k)


def sel_l1(ctx, k):
    return l1_path_order(ctx.Ztr[ctx.sub_l1], ctx.ytr[ctx.sub_l1], k,
                         ctx.seed, L1_C_GRID, L1_MAX_ITER)


def sel_correlation(ctx, k):
    """max_c |Pearson r(f, 1{y = c})| -- the multiclass form of a correlation filter."""
    Z = ctx.Ztr
    Zc = Z - Z.mean(axis=0)
    zs = np.sqrt((Zc ** 2).sum(axis=0));  zs[zs == 0] = 1.0
    Y = np.zeros((len(ctx.ytr), len(np.unique(ctx.ytr))))
    Y[np.arange(len(ctx.ytr)), ctx.ytr] = 1.0
    Yc = Y - Y.mean(axis=0)
    ys = np.sqrt((Yc ** 2).sum(axis=0));  ys[ys == 0] = 1.0
    corr = (Zc.T @ Yc) / np.outer(zs, ys)
    return order_from_scores(np.abs(corr).max(axis=1), k)


def sel_anova(ctx, k):
    f, _ = f_classif(ctx.Ztr, ctx.ytr)
    return order_from_scores(f, k)


def sel_chi2(ctx, k):
    stat, _ = chi2(ctx.Xtr_mm, ctx.ytr)          # requires non-negative input
    return order_from_scores(stat, k)


def sel_mrmr(ctx, k):
    return mrmr(ctx.mi, ctx.codes, k)[0]


SELECTORS = {
    "CGFS-weighted":   sel_cgfs_weighted,
    "CGFS-unweighted": sel_cgfs_unweighted,
    "Random":          sel_random,
    "RFE":             sel_rfe,
    "XGBoost":         sel_xgboost,
    "MutualInfo":      sel_mutual_info,
    "ReliefF":         sel_relieff,
    "L1":              sel_l1,
    "Correlation":     sel_correlation,
    "ANOVA":           sel_anova,
    "Chi2":            sel_chi2,
}
if INCLUDE_MRMR:
    SELECTORS["mRMR"] = sel_mrmr

METHOD_ORDER = list(SELECTORS.keys())

# Which shared prep component each selector consumes without timing it itself.
# The run loop charges these back into select_total_s.
SELECTOR_PREP = {
    "CGFS-weighted":   ("sim", "mi"),   # needs the similarity matrix and the weights
    "CGFS-unweighted": ("sim",),
}
if INCLUDE_MRMR:
    SELECTOR_PREP["mRMR"] = ("mi",)     # relevance scores + the binned columns
if MI_BASELINE == "plugin":
    SELECTOR_PREP["MutualInfo"] = ("mi",)   # in "sklearn" mode it scores inline


def prep_charge(method, ctx):
    """Shared prep seconds attributable to this selector."""
    parts = SELECTOR_PREP.get(method, ())
    return (ctx.prep_sim_s if "sim" in parts else 0.0) + \
           (ctx.prep_mi_s if "mi" in parts else 0.0)


print(f"{len(SELECTORS)} selectors:", METHOD_ORDER)
print("prep charged back to:", {k: "+".join(v) for k, v in SELECTOR_PREP.items()})

12 selectors: ['CGFS-weighted', 'CGFS-unweighted', 'Random', 'RFE', 'XGBoost', 'MutualInfo', 'ReliefF', 'L1', 'Correlation', 'ANOVA', 'Chi2', 'mRMR']
prep charged back to: {'CGFS-weighted': 'sim+mi', 'CGFS-unweighted': 'sim', 'mRMR': 'mi'}


## 11. Classifiers and downstream evaluation

All five classifiers train on `Z`, as the script does.

* **`rf`** — the original random forest, unchanged.
* **`dt`** — a single unpruned decision tree.
* **`gb`** — gradient boosting in its *fast histogram* configuration: XGBoost
  with `tree_method="hist"` when the package imports, otherwise scikit-learn's
  `HistGradientBoostingClassifier`, which bins features the same way. The exact
  `GradientBoostingClassifier` is deliberately not used — it is roughly an order
  of magnitude slower and would dominate the runtime of the whole grid.
* **`svm`** — `LinearSVC` by default. An RBF `SVC` is O(n²) in the training set
  and will not finish on a full IoT capture, so `SVM_KIND = "rbf"` also caps the
  training set at `SVM_RBF_MAX_TRAIN` rows. That cap makes RBF results *not*
  directly comparable to the other classifiers, which all see the full split —
  worth stating explicitly if you report them.
* **`knn`** — `CustomKNN` below, brute-force k-NN written out rather than taken
  from `sklearn.neighbors`. `KNN_MAX_TRAIN` caps the reference set; with it set
  to `None` the distance matrix is `n_test × n_train` per budget, which is the
  single most expensive thing in this notebook.

**Metrics.** Precision, recall and F1 are support-weighted; macro-F1 is separate
so minority attack classes carry equal weight. `FPR` is the macro-averaged
per-class false-positive rate `FP / (FP + TN)` taken from the confusion matrix —
in a multiclass setting a "false positive" for class *c* is any other-class flow
predicted as *c*. `fpr_weighted` is also stored in the raw results, and setting
`NORMAL_CLASS` adds `fpr_binary`, the fraction of true-benign flows flagged as
some attack, which is the false-alarm rate an IDS paper usually quotes.
`NORMAL_CLASS` takes either one class name or a list of them, since RT-IoT2022
splits benign traffic across several classes; a benign flow predicted as a
different benign class does not count as a false alarm.

**Timing.** `fit_time_s` and `inference_time_s` are measured with
`perf_counter` around `fit` and `predict`; inference is over the whole test
split, and `inference_us_per_sample` normalises it.

In [11]:
class CustomKNN:
    """Brute-force k-nearest-neighbours, written out rather than imported.

    The distance block is one BLAS matmul per chunk: ranking by ||q - r||^2 is
    the same as ranking by ||r||^2 - 2 q.r, so the ||q||^2 term is never formed.
    Only the k nearest are materialised, via argpartition rather than a sort.
    """

    def __init__(self, n_neighbors=5, weights="distance", chunk=512,
                 max_train=None, random_state=0):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.chunk = chunk
        self.max_train = max_train
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        if self.max_train is not None and self.max_train < len(y):
            rng = np.random.default_rng(self.random_state)
            idx = stratified_subsample(y, self.max_train, rng)
        else:
            idx = np.arange(len(y))
        self.R_ = np.ascontiguousarray(X[idx], dtype=np.float32)
        self.y_ = y[idx]
        self.r_sq_ = (self.R_ ** 2).sum(axis=1)
        self.classes_ = np.unique(self.y_)
        self.k_ = int(min(self.n_neighbors, len(self.y_)))
        return self

    def predict(self, X):
        Q = np.ascontiguousarray(X, dtype=np.float32)
        out = np.empty(len(Q), dtype=self.y_.dtype)
        for s in range(0, len(Q), self.chunk):
            q = Q[s:s + self.chunk]
            D = self.r_sq_[None, :] - 2.0 * (q @ self.R_.T)     # rank-equivalent to ||q-r||^2
            nb = np.argpartition(D, self.k_ - 1, axis=1)[:, :self.k_]
            lab = self.y_[nb]
            if self.weights == "distance":
                dsq = np.take_along_axis(D, nb, axis=1) + (q ** 2).sum(axis=1, keepdims=True)
                wgt = 1.0 / np.sqrt(np.maximum(dsq, 1e-12))
                wgt[~np.isfinite(wgt)] = 1e12                   # exact duplicates dominate
            else:
                wgt = np.ones_like(lab, dtype=np.float32)
            votes = np.stack([(wgt * (lab == c)).sum(axis=1) for c in self.classes_], axis=1)
            out[s:s + len(q)] = self.classes_[votes.argmax(axis=1)]
        return out


def make_gb(seed):
    """Gradient boosting, fast histogram configuration (never the exact splitter)."""
    if GB_BACKEND in ("auto", "xgboost") and HAVE_XGB:
        return XGBClassifier(n_estimators=GB_N_ESTIMATORS, max_depth=GB_MAX_DEPTH,
                             learning_rate=GB_LR, tree_method="hist",
                             max_bin=GB_MAX_BINS, n_jobs=N_JOBS,
                             random_state=seed, verbosity=0)
    return HistGradientBoostingClassifier(
        max_iter=GB_N_ESTIMATORS, learning_rate=GB_LR,
        max_bins=min(GB_MAX_BINS, 255), early_stopping=True,
        n_iter_no_change=10, validation_fraction=0.1, random_state=seed)


def make_svm(seed):
    if SVM_KIND == "rbf":
        # Kernel SVM is O(n^2); the cap is what makes it finish at all.
        return SVC(C=SVM_C, kernel="rbf", gamma="scale", cache_size=1000,
                   random_state=seed)
    if SVM_KIND == "sgd":
        # A linear SVM by a different route: identical hinge loss and L2 penalty,
        # solved by stochastic gradient descent instead of liblinear's Newton
        # method. Cost is linear in n_train rather than superlinear, and on
        # standardised inputs the boundary lands in essentially the same place.
        return SGDClassifier(loss="hinge", penalty="l2", alpha=SVM_SGD_ALPHA,
                             max_iter=SVM_SGD_MAX_ITER, tol=SVM_TOL,
                             early_stopping=True, n_iter_no_change=5,
                             validation_fraction=0.1, n_jobs=N_JOBS,
                             random_state=seed)
    base = LinearSVC(C=SVM_C, dual=False, tol=SVM_TOL, max_iter=SVM_MAX_ITER,
                     random_state=seed)
    # LinearSVC already reduces multiclass to one-vs-rest, but inside liblinear,
    # sequentially. OneVsRestClassifier fits the same per-class problems in
    # parallel: identical decision rule, wall-clock divided by the core count.
    return OneVsRestClassifier(base, n_jobs=N_JOBS) if SVM_PARALLEL_OVR else base


CLASSIFIERS = {
    "rf":     lambda seed: RandomForestClassifier(n_estimators=CLF_N_ESTIMATORS,
                                                  n_jobs=N_JOBS, random_state=seed),
    "dt":     lambda seed: DecisionTreeClassifier(random_state=seed),
    "gb":     make_gb,
    "svm":    make_svm,
    "knn":    lambda seed: CustomKNN(n_neighbors=KNN_NEIGHBORS, weights=KNN_WEIGHTS,
                                     chunk=KNN_CHUNK, max_train=KNN_MAX_TRAIN,
                                     random_state=seed),
    "logreg": lambda seed: LogisticRegression(max_iter=1000, n_jobs=N_JOBS,
                                              random_state=seed),
}

# The RBF cap breaks comparability with the other classifiers; say so loudly.
if SVM_KIND == "rbf" and "svm" in CLASSIFIERS_TO_RUN:
    print(f"NOTE: svm = RBF SVC trained on at most {SVM_RBF_MAX_TRAIN:,} rows, "
          f"unlike every other classifier here, which sees the full split.")

LABELS_ALL = np.arange(len(LE_Y.classes_))


def _encode_normal(spec_):
    """NORMAL_CLASS may be one class name or several; return their encoded labels."""
    if spec_ is None:
        return None
    names = [spec_] if isinstance(spec_, str) else list(spec_)
    known = {str(c) for c in LE_Y.classes_}
    missing = [n for n in names if str(n) not in known]
    if missing:
        raise ValueError(f"NORMAL_CLASS entries not present in this dataset: {missing}. "
                         f"Available classes: {sorted(known)}")
    return np.sort(LE_Y.transform(names)).astype(int)


NORMAL_LABELS = _encode_normal(NORMAL_CLASS)
if NORMAL_LABELS is not None:
    print("benign classes for the binary false-alarm rate:",
          [str(c) for c in LE_Y.inverse_transform(NORMAL_LABELS)])

METRICS = ["accuracy", "precision", "recall", "f1", "macro_f1", "fpr"]
if NORMAL_LABELS is not None:
    METRICS.append("fpr_binary")

TIME_COLS = ["select_time_s", "select_total_s", "fit_time_s", "inference_time_s"]
REPORT_COLS = METRICS + TIME_COLS

METRIC_LABELS = {
    "accuracy": "Acc", "precision": "Prec(w)", "recall": "Rec(w)",
    "f1": "F1(w)", "macro_f1": "Macro-F1", "fpr": "FPR",
    "fpr_binary": "FPR(bin)", "fpr_weighted": "FPR(w)",
    "select_time_s": "Select(s)", "select_total_s": "Select+prep(s)",
    "fit_time_s": "Fit(s)", "inference_time_s": "Infer(s)",
}


def fpr_from_confusion(y_true, y_pred):
    """Macro- and support-weighted per-class FPR = FP / (FP + TN)."""
    cm = confusion_matrix(y_true, y_pred, labels=LABELS_ALL)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    tn = cm.sum() - (tp + fp + fn)
    per_class = fp / np.maximum(fp + tn, 1.0)
    support = cm.sum(axis=1)
    present = support > 0                       # classes absent from the test split
    if not present.any():
        return float("nan"), float("nan")
    return (float(per_class[present].mean()),
            float(np.average(per_class[present], weights=support[present])))


def evaluate_subset(ctx, feat_idx, clf_name):
    idx = list(feat_idx)
    clf = CLASSIFIERS[clf_name](ctx.seed)

    t0 = time.perf_counter()
    clf.fit(ctx.Ztr[:, idx], ctx.ytr)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    pred = clf.predict(ctx.Zte[:, idx])
    infer_time = time.perf_counter() - t0

    yt = ctx.yte
    fpr_macro, fpr_weighted = fpr_from_confusion(yt, pred)
    out = {
        "accuracy":     accuracy_score(yt, pred),
        "precision":    precision_score(yt, pred, average="weighted", zero_division=0),
        "recall":       recall_score(yt, pred, average="weighted", zero_division=0),
        "f1":           f1_score(yt, pred, average="weighted", zero_division=0),
        "macro_f1":     f1_score(yt, pred, average="macro", zero_division=0),
        "fpr":          fpr_macro,
        "fpr_weighted": fpr_weighted,
        "fit_time_s":       fit_time,
        "inference_time_s": infer_time,
        "inference_us_per_sample": 1e6 * infer_time / max(len(yt), 1),
    }
    if NORMAL_LABELS is not None:
        # False alarm = a genuinely benign flow predicted as some attack. Benign
        # flows confused with another benign class are not false alarms.
        mask = np.isin(yt, NORMAL_LABELS)
        out["fpr_binary"] = (float((~np.isin(pred[mask], NORMAL_LABELS)).mean())
                             if mask.any() else np.nan)
    return out

benign classes for the binary false-alarm rate: ['MQTT_Publish', 'Thing_Speak', 'Wipro_bulb']


### 11a. SVM cost check (optional, but run it before a long grid)

`evaluate_subset` fits the SVM 72 times per classifier pass, so anything wasted
per fit is multiplied by 72. Two things commonly waste it:

* **Non-convergence.** Cell 2 sets `warnings.filterwarnings("ignore")`, which
  silences `ConvergenceWarning`. A `LinearSVC` that never reaches `tol` runs the
  full `max_iter` on every one of those 72 fits and says nothing about it. The
  check below re-enables the warning so you can see whether that is happening.
* **Serial one-vs-rest.** RT-IoT2022 has a dozen classes, and liblinear walks
  them one at a time on a single core.

The cell times three configurations on one real feature subset and reports
accuracy and macro-F1 for each, so the choice is made on your data rather than on
a rule of thumb. Nothing here changes the experiment — set `SVM_KIND` and
`SVM_PARALLEL_OVR` in cell 2 once you have picked.

In [12]:
# Times three linear-SVM configurations on one subset. Cheap: 3 fits, not 72.
# _ctx = build_context(SEEDS[0])
# _k = min(30, _ctx.d)
# _idx = order_from_scores(f_classif(_ctx.Ztr, _ctx.ytr)[0], _k)   # ANOVA top-k, cheap
# _Xtr, _Xte = _ctx.Ztr[:, _idx], _ctx.Zte[:, _idx]

# _variants = {
#     "LinearSVC serial   ": LinearSVC(C=SVM_C, dual=False, tol=SVM_TOL,
#                                      max_iter=SVM_MAX_ITER, random_state=SEEDS[0]),
#     "LinearSVC parallel ": OneVsRestClassifier(
#         LinearSVC(C=SVM_C, dual=False, tol=SVM_TOL, max_iter=SVM_MAX_ITER,
#                   random_state=SEEDS[0]), n_jobs=N_JOBS),
#     "SGD hinge          ": SGDClassifier(loss="hinge", penalty="l2", alpha=SVM_SGD_ALPHA,
#                                          max_iter=SVM_SGD_MAX_ITER, tol=SVM_TOL,
#                                          early_stopping=True, n_iter_no_change=5,
#                                          validation_fraction=0.1, n_jobs=N_JOBS,
#                                          random_state=SEEDS[0]),
# }

# print(f"one fit on d={_k} features, {_Xtr.shape[0]:,} train / {_Xte.shape[0]:,} test\n")
# print(f"{'variant':21s} {'fit s':>8s} {'infer s':>8s} {'Acc':>8s} {'Macro-F1':>9s}  convergence")
# for _name, _clf in _variants.items():
#     with warnings.catch_warnings(record=True) as _w:
#         warnings.simplefilter("always")          # cell 2 turned these off globally
#         _t0 = time.perf_counter(); _clf.fit(_Xtr, _ctx.ytr); _fit = time.perf_counter() - _t0
#         _hit = any("onvergence" in str(x.message) for x in _w)
#     _t0 = time.perf_counter(); _p = _clf.predict(_Xte); _inf = time.perf_counter() - _t0
#     _note = "HIT max_iter" if _hit else "converged"
#     print(f"{_name:21s} {_fit:8.2f} {_inf:8.2f} "
#           f"{accuracy_score(_ctx.yte, _p):8.4f} {f1_score(_ctx.yte, _p, average='macro', zero_division=0):9.4f}"
#           f"  {_note}")

# print(f"\nAt 72 fits per seed, one second saved per fit is {72 / 60:.1f} min per seed.")
# print("If a variant HIT max_iter, that fit burned the full budget every time: lower")
# print("SVM_C (stronger regularisation converges faster) or raise SVM_TOL before")
# print("raising SVM_MAX_ITER, and re-check accuracy here.")

## 12. Run the experiment

The grid is `len(SELECTORS) × len(K_LIST) × len(CLASSIFIERS_TO_RUN) × len(SEEDS)`
model fits, plus one selection run per selector per budget per seed. The cell
prints that count before it starts.

In [13]:
records, full_records = [], []
KMAX = max(K_LIST)


def _take(order, k, name):
    out = list(dict.fromkeys(int(i) for i in order))[:k]
    if len(out) < k:
        raise RuntimeError(f"{name} returned {len(out)} distinct features, needed {k}")
    return out


n_fits = len(SELECTORS) * len(K_LIST) * len(CLASSIFIERS_TO_RUN) * len(SEEDS)
print(f"{len(SELECTORS)} selectors x {len(K_LIST)} budgets x "
      f"{len(CLASSIFIERS_TO_RUN)} classifiers x {len(SEEDS)} seeds "
      f"= {n_fits} model fits, plus a full-feature ceiling per classifier/seed")
n_sel = len(SELECTORS) * (len(K_LIST) if SELECT_PER_K else 1) * len(SEEDS)
print(f"selection runs: {n_sel}")

# mutual_info_classif runs a nearest-neighbour search per feature, so on a full
# split it is the second-slowest selector here after RFE.
if SELECT_PER_K and MI_BASELINE == "sklearn":
    rows = "the full training split" if MI_SUBSAMPLE is None else f"{MI_SUBSAMPLE:,} rows"
    print(f"\nNOTE: mutual_info_classif will run {len(K_LIST) * len(SEEDS)} times "
          f"({len(K_LIST)} budgets x {len(SEEDS)} seeds) on {rows}.")
    print("      Its score does not depend on k, so every budget after the first")
    print("      recomputes the same vector. Set MI_SUBSAMPLE to e.g. 20_000 if that")
    print("      is too slow; SELECT_PER_K=False also collapses it to one run per seed.")

# RFE is the one selector whose cost is measured in minutes rather than seconds,
# and per-budget selection multiplies it by len(K_LIST).
if SELECT_PER_K and "RFE" in SELECTORS:
    n_rfe = len(K_LIST) * len(SEEDS)
    rows = "the full training split" if RFE_SUBSAMPLE is None else f"{RFE_SUBSAMPLE:,} rows"
    print(f"\nNOTE: RFE will run {n_rfe} times ({len(K_LIST)} budgets x {len(SEEDS)} seeds) "
          f"on {rows}.")
    if RFE_SUBSAMPLE is None:
        print("      On a full IoT capture that is hours, not minutes. Set RFE_SUBSAMPLE")
        print("      to e.g. 20_000, raise RFE_STEP, or set SELECT_PER_K=False (one run at")
        print(f"      k={max(K_LIST)} with the prefixes reused) if that is too slow.")

t_start = time.time()

for seed in SEEDS:
    print(f"\n{'=' * 78}\nSEED {seed}\n{'=' * 78}", flush=True)
    ctx = build_context(seed)
    print(f"  screening: {D_TOTAL - ctx.d} constant/near-constant columns removed, d={ctx.d}")
    print(f"  weights: min={ctx.w.min():.4f} max={ctx.w.max():.4f} "
          f"zeros={(ctx.w < 1e-9).sum()}  (sum={ctx.w.sum():.2f}, d={ctx.d})")
    print(f"  sim + weights (prep) built in {ctx.prep_s:.2f}s")
    print(f"  train {ctx.Ztr.shape[0]:,} / test {ctx.Zte.shape[0]:,}")

    budgets = [k for k in K_LIST if k <= ctx.d]

    # ---- selection, timed at each budget -----------------------------------
    picks = {}
    print("  selection:")
    for name, fn in SELECTORS.items():
        if SELECT_PER_K:
            for k in budgets:
                t0 = time.perf_counter()
                order = fn(ctx, k)
                picks[(name, k)] = (_take(order, k, name), time.perf_counter() - t0)
        else:
            t0 = time.perf_counter()
            order = fn(ctx, KMAX)
            elapsed = time.perf_counter() - t0
            for k in budgets:
                picks[(name, k)] = (_take(order, k, name), elapsed)
        charge = prep_charge(name, ctx)
        times = [picks[(name, k)][1] for k in budgets]
        if VERBOSE:
            extra = f" +{charge:.2f}s prep" if charge > 0 else ""
            print(f"    {name:<16} {sum(times):7.2f}s total "
                  f"({min(times):.2f}-{max(times):.2f}s per budget){extra}  "
                  f"top5 = {[ctx.feat_names[i] for i in picks[(name, budgets[0])][0][:5]]}")

    # ---- evaluation ---------------------------------------------------------
    for clf_name in CLASSIFIERS_TO_RUN:
        t_clf = time.time()
        m = evaluate_subset(ctx, range(ctx.d), clf_name)
        full_records.append({"seed": seed, "classifier": clf_name, "k": ctx.d,
                             **{key: m[key] for key in METRICS if key in m},
                             "fit_time_s": m["fit_time_s"],
                             "inference_time_s": m["inference_time_s"]})

        for (name, k), (idx, sel_t) in picks.items():
            m = evaluate_subset(ctx, idx, clf_name)
            records.append({
                "seed": seed, "classifier": clf_name, "method": name, "k": k,
                **{key: m[key] for key in METRICS},
                "fpr_weighted": m["fpr_weighted"],
                "select_time_s": sel_t,
                "select_total_s": sel_t + prep_charge(name, ctx),
                "fit_time_s": m["fit_time_s"],
                "inference_time_s": m["inference_time_s"],
                "inference_us_per_sample": m["inference_us_per_sample"],
                "prep_time_s": ctx.prep_s,
                "prep_sim_s": ctx.prep_sim_s,
                "prep_mi_s": ctx.prep_mi_s,
                "d_after_screening": ctx.d,
                "features": "|".join(str(ctx.feat_names[i]) for i in idx),
            })
        print(f"  [{clf_name}] {len(picks)} fits in {time.time() - t_clf:.1f}s", flush=True)

raw = pd.DataFrame(records)
full = pd.DataFrame(full_records)
print(f"\nDone in {(time.time() - t_start) / 60:.1f} min -> {len(raw)} rows")
raw.head()

12 selectors x 6 budgets x 5 classifiers x 3 seeds = 1080 model fits, plus a full-feature ceiling per classifier/seed
selection runs: 216

NOTE: mutual_info_classif will run 18 times (6 budgets x 3 seeds) on the full training split.
      Its score does not depend on k, so every budget after the first
      recomputes the same vector. Set MI_SUBSAMPLE to e.g. 20_000 if that
      is too slow; SELECT_PER_K=False also collapses it to one run per seed.

NOTE: RFE will run 18 times (6 budgets x 3 seeds) on the full training split.
      On a full IoT capture that is hours, not minutes. Set RFE_SUBSAMPLE
      to e.g. 20_000, raise RFE_STEP, or set SELECT_PER_K=False (one run at
      k=30 with the prefixes reused) if that is too slow.

SEED 0
  screening: 6 constant/near-constant columns removed, d=77
  weights: min=0.0000 max=1.8351 zeros=5  (sum=77.00, d=77)
  sim + weights (prep) built in 0.56s
  train 86,181 / test 36,936
  selection:
    CGFS-weighted       0.01s total (0.00-0.00s per

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda

  [svm] 72 fits in 1590.1s
  [knn] 72 fits in 646.0s

SEED 1
  screening: 6 constant/near-constant columns removed, d=77
  weights: min=0.0000 max=1.8393 zeros=5  (sum=77.00, d=77)
  sim + weights (prep) built in 0.54s
  train 86,181 / test 36,936
  selection:
    CGFS-weighted       0.01s total (0.00-0.00s per budget) +0.54s prep  top5 = ['service', 'fwd_header_size_tot', 'flow_iat.avg', 'payload_bytes_per_second', 'flow_pkts_payload.max']
    CGFS-unweighted     0.01s total (0.00-0.00s per budget) +0.06s prep  top5 = ['service', 'fwd_header_size_tot', 'flow_iat.std', 'bwd_header_size_min', 'flow_pkts_payload.max']
    Random              0.00s total (0.00-0.00s per budget)  top5 = ['idle.min', 'fwd_header_size_tot', 'flow_pkts_payload.tot', 'fwd_subflow_pkts', 'fwd_iat.tot']
    RFE               307.21s total (43.08-58.59s per budget)  top5 = ['id.resp_p', 'fwd_pkts_payload.min', 'fwd_pkts_payload.avg', 'fwd_subflow_bytes', 'active.min']
    XGBoost            92.33s total (15.03-15

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda

  [svm] 72 fits in 1379.3s
  [knn] 72 fits in 583.0s

SEED 2
  screening: 6 constant/near-constant columns removed, d=77
  weights: min=0.0000 max=1.8388 zeros=5  (sum=77.00, d=77)
  sim + weights (prep) built in 0.51s
  train 86,181 / test 36,936
  selection:
    CGFS-weighted       0.01s total (0.00-0.00s per budget) +0.51s prep  top5 = ['fwd_init_window_size', 'fwd_header_size_tot', 'flow_pkts_payload.std', 'flow_iat.avg', 'flow_pkts_per_sec']
    CGFS-unweighted     0.00s total (0.00-0.00s per budget) +0.06s prep  top5 = ['service', 'fwd_header_size_tot', 'flow_iat.std', 'bwd_header_size_min', 'flow_iat.tot']
    Random              0.00s total (0.00-0.00s per budget)  top5 = ['bwd_header_size_tot', 'bwd_pkts_payload.avg', 'fwd_iat.avg', 'proto', 'fwd_iat.max']
    RFE               312.81s total (44.37-59.28s per budget)  top5 = ['id.resp_p', 'service', 'fwd_pkts_payload.min', 'fwd_pkts_payload.avg', 'flow_iat.min']
    XGBoost            91.73s total (14.78-16.34s per budget)  to

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/zeus/miniconda

  [svm] 72 fits in 1331.7s
  [knn] 72 fits in 612.4s

Done in 155.2 min -> 1080 rows


,seed,classifier,method,k,accuracy,precision,recall,f1,macro_f1,fpr,...,select_time_s,select_total_s,fit_time_s,inference_time_s,inference_us_per_sample,prep_time_s,prep_sim_s,prep_mi_s,d_after_screening,features
0,0,rf,CGFS-weighted,5,0.958117,0.968579,0.958117,0.959358,0.809779,0.003575,...,0.000723,0.561655,0.974401,0.096759,2.619644,0.560931,0.076437,0.484494,77,fwd_init_window_size|fwd_header_size_tot|flow_...
1,0,rf,CGFS-weighted,10,0.978151,0.978544,0.978151,0.977780,0.847659,0.001867,...,0.000790,0.561722,1.051636,0.126529,3.425638,0.560931,0.076437,0.484494,77,fwd_init_window_size|fwd_header_size_tot|flow_...
2,0,rf,CGFS-weighted,15,0.978882,0.978838,0.978882,0.978839,0.859386,0.001802,...,0.000817,0.561748,1.048246,0.095904,2.596502,0.560931,0.076437,0.484494,77,fwd_init_window_size|fwd_header_size_tot|flow_...
3,0,rf,CGFS-weighted,20,0.979397,0.979359,0.979397,0.979359,0.862314,0.001757,...,0.000924,0.561855,1.233133,0.137413,3.720298,0.560931,0.076437,0.484494,77,fwd_init_window_size|fwd_header_size_tot|flow_...
4,0,rf,CGFS-weighted,25,0.997915,0.997920,0.997915,0.997911,0.965137,0.000185,...,0.000957,0.561888,1.520471,0.096822,2.621348,0.560931,0.076437,0.484494,77,fwd_init_window_size|fwd_header_size_tot|flow_...


## 13. Aggregate to mean ± std

In [14]:
def fmt(mean, std, nd=4, pm="\u00b1"):
    if pd.isna(std):                       # a single seed -> no spread to report
        return f"{mean:.{nd}f}"
    return f"{mean:.{nd}f} {pm} {std:.{nd}f}"


def _nd(col):
    return 3 if col in TIME_COLS else 4     # times in ms-ish precision, rates in 4dp


agg = (raw.groupby(["classifier", "method", "k"])[REPORT_COLS]
          .agg(["mean", "std"]).reset_index())
agg.columns = ["classifier", "method", "k"] + [f"{m}_{s}" for m in REPORT_COLS
                                               for s in ("mean", "std")]
agg["method"] = pd.Categorical(agg["method"], categories=METHOD_ORDER, ordered=True)
agg = agg.sort_values(["classifier", "k", "method"]).reset_index(drop=True)

summary     = agg[["classifier", "method", "k"]].copy()
summary_tex = agg[["classifier", "method", "k"]].copy()
for m in REPORT_COLS:
    pairs = list(zip(agg[f"{m}_mean"], agg[f"{m}_std"]))
    summary[METRIC_LABELS[m]]     = [fmt(mu, sd, _nd(m)) for mu, sd in pairs]
    summary_tex[METRIC_LABELS[m]] = [fmt(mu, sd, _nd(m), pm=r"$\pm$") for mu, sd in pairs]

# Full-feature ceiling, for reference.
ceil_cols = [c for c in METRICS if c in full.columns] + ["fit_time_s", "inference_time_s"]
full_agg = full.groupby("classifier")[ceil_cols].agg(["mean", "std"])
print("Full feature set (ceiling)\n")
for clf_name in CLASSIFIERS_TO_RUN:
    row = " | ".join(
        f"{METRIC_LABELS[m]} {fmt(full_agg.loc[clf_name, (m, 'mean')], full_agg.loc[clf_name, (m, 'std')], _nd(m))}"
        for m in ceil_cols)
    d_here = int(full[full.classifier == clf_name]["k"].mean())
    print(f"  {clf_name:<7} d={d_here}  {row}")

summary.head(len(METHOD_ORDER))

Full feature set (ceiling)

  rf      d=77  Acc 0.9981 ± 0.0005 | Prec(w) 0.9981 ± 0.0005 | Rec(w) 0.9981 ± 0.0005 | F1(w) 0.9981 ± 0.0005 | Macro-F1 0.9738 ± 0.0101 | FPR 0.0002 ± 0.0000 | FPR(bin) 0.0076 ± 0.0033 | Fit(s) 1.854 ± 0.047 | Infer(s) 0.113 ± 0.017
  dt      d=77  Acc 0.9973 ± 0.0003 | Prec(w) 0.9974 ± 0.0003 | Rec(w) 0.9973 ± 0.0003 | F1(w) 0.9973 ± 0.0003 | Macro-F1 0.9488 ± 0.0102 | FPR 0.0002 ± 0.0000 | FPR(bin) 0.0099 ± 0.0010 | Fit(s) 0.602 ± 0.035 | Infer(s) 0.008 ± 0.000
  gb      d=77  Acc 0.9977 ± 0.0004 | Prec(w) 0.9977 ± 0.0004 | Rec(w) 0.9977 ± 0.0004 | F1(w) 0.9977 ± 0.0004 | Macro-F1 0.9673 ± 0.0173 | FPR 0.0002 ± 0.0000 | FPR(bin) 0.0067 ± 0.0008 | Fit(s) 8.021 ± 0.192 | Infer(s) 0.140 ± 0.024
  svm     d=77  Acc 0.9825 ± 0.0018 | Prec(w) 0.9826 ± 0.0018 | Rec(w) 0.9825 ± 0.0018 | F1(w) 0.9823 ± 0.0019 | Macro-F1 0.8763 ± 0.0383 | FPR 0.0016 ± 0.0002 | FPR(bin) 0.0548 ± 0.0063 | Fit(s) 162.267 ± 68.967 | Infer(s) 0.036 ± 0.011
  knn     d=77  Acc 0.9945 ± 

,classifier,method,k,Acc,Prec(w),Rec(w),F1(w),Macro-F1,FPR,FPR(bin),Select(s),Select+prep(s),Fit(s),Infer(s)
0,dt,CGFS-weighted,5,0.9656 ± 0.0126,0.9725 ± 0.0070,0.9656 ± 0.0126,0.9663 ± 0.0117,0.8192 ± 0.0301,0.0029 ± 0.0011,0.0347 ± 0.0064,0.001 ± 0.000,0.537 ± 0.024,0.044 ± 0.001,0.002 ± 0.000
1,dt,CGFS-unweighted,5,0.9614 ± 0.0138,0.9379 ± 0.0270,0.9614 ± 0.0138,0.9489 ± 0.0199,0.7219 ± 0.0848,0.0093 ± 0.0064,0.0613 ± 0.0349,0.001 ± 0.000,0.067 ± 0.008,0.034 ± 0.009,0.002 ± 0.000
2,dt,Random,5,0.9489 ± 0.0237,0.9260 ± 0.0373,0.9489 ± 0.0237,0.9336 ± 0.0321,0.6375 ± 0.1111,0.0132 ± 0.0098,0.0281 ± 0.0037,0.000 ± 0.000,0.000 ± 0.000,0.031 ± 0.008,0.002 ± 0.000
3,dt,RFE,5,0.9704 ± 0.0001,0.9704 ± 0.0001,0.9704 ± 0.0001,0.9703 ± 0.0001,0.7823 ± 0.0072,0.0025 ± 0.0000,0.0205 ± 0.0017,59.439 ± 0.943,59.439 ± 0.943,0.041 ± 0.008,0.002 ± 0.000
4,dt,XGBoost,5,0.9821 ± 0.0029,0.9805 ± 0.0052,0.9821 ± 0.0029,0.9803 ± 0.0045,0.7624 ± 0.0387,0.0015 ± 0.0002,0.0294 ± 0.0045,15.175 ± 0.412,15.175 ± 0.412,0.023 ± 0.009,0.002 ± 0.000
5,dt,MutualInfo,5,0.9552 ± 0.0007,0.9608 ± 0.0053,0.9552 ± 0.0007,0.9510 ± 0.0007,0.6200 ± 0.0149,0.0038 ± 0.0001,0.0230 ± 0.0029,19.125 ± 0.085,19.125 ± 0.085,0.023 ± 0.001,0.002 ± 0.000
6,dt,ReliefF,5,0.9462 ± 0.0324,0.9490 ± 0.0291,0.9462 ± 0.0324,0.9396 ± 0.0417,0.6247 ± 0.0885,0.0084 ± 0.0093,0.1482 ± 0.1963,0.758 ± 0.022,0.758 ± 0.022,0.042 ± 0.006,0.002 ± 0.000
7,dt,L1,5,0.9832 ± 0.0008,0.9839 ± 0.0048,0.9832 ± 0.0008,0.9801 ± 0.0009,0.7558 ± 0.0097,0.0014 ± 0.0001,0.0370 ± 0.0045,19.646 ± 7.439,19.646 ± 7.439,0.016 ± 0.001,0.002 ± 0.000
8,dt,Correlation,5,0.9753 ± 0.0007,0.9796 ± 0.0006,0.9753 ± 0.0007,0.9758 ± 0.0007,0.7542 ± 0.0100,0.0021 ± 0.0001,0.1180 ± 0.0028,0.047 ± 0.005,0.047 ± 0.005,0.018 ± 0.001,0.002 ± 0.000
9,dt,ANOVA,5,0.9915 ± 0.0021,0.9916 ± 0.0020,0.9915 ± 0.0021,0.9914 ± 0.0021,0.8622 ± 0.0174,0.0007 ± 0.0002,0.0308 ± 0.0042,0.102 ± 0.010,0.102 ± 0.010,0.018 ± 0.002,0.002 ± 0.000


## 14. Tables

Three views of the same numbers: one table per (classifier, budget) with every
metric and every timing side by side; one table per metric across budgets; and
the `ΔF1 vs random` / `z` block from the script.

In [15]:
pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 80)

for clf_name in CLASSIFIERS_TO_RUN:
    sub_c = summary[summary.classifier == clf_name]
    for k in sorted(sub_c["k"].unique()):
        block = sub_c[sub_c["k"] == k].drop(columns=["k", "classifier"]).set_index("method")
        block.index.name = None
        print(f"\n=== {clf_name} | top-{k} features " + "=" * 46)
        print(block.to_string())


=== rf | top-5 features ==============================================
                             Acc          Prec(w)           Rec(w)            F1(w)         Macro-F1              FPR         FPR(bin)       Select(s)  Select+prep(s)         Fit(s)       Infer(s)
CGFS-weighted    0.9664 ± 0.0129  0.9734 ± 0.0072  0.9664 ± 0.0129  0.9671 ± 0.0120  0.8365 ± 0.0313  0.0029 ± 0.0011  0.0324 ± 0.0086   0.001 ± 0.000   0.537 ± 0.024  0.903 ± 0.066  0.095 ± 0.001
CGFS-unweighted  0.9626 ± 0.0140  0.9391 ± 0.0271  0.9626 ± 0.0140  0.9501 ± 0.0200  0.7345 ± 0.0931  0.0092 ± 0.0064  0.0591 ± 0.0360   0.001 ± 0.000   0.067 ± 0.008  0.792 ± 0.097  0.109 ± 0.016
Random           0.9493 ± 0.0235  0.9264 ± 0.0371  0.9493 ± 0.0235  0.9341 ± 0.0319  0.6403 ± 0.1089  0.0131 ± 0.0098  0.0282 ± 0.0045   0.000 ± 0.000   0.000 ± 0.000  0.726 ± 0.103  0.109 ± 0.015
RFE              0.9716 ± 0.0002  0.9715 ± 0.0003  0.9716 ± 0.0002  0.9715 ± 0.0003  0.7943 ± 0.0079  0.0024 ± 0.0000  0.0187 ± 0.0024  59.4

In [16]:
def pivot_metric(frame, m, clf_name):
    sub_c = frame[frame.classifier == clf_name]
    tbl = sub_c.pivot(index="method", columns="k", values=METRIC_LABELS[m]).reindex(METHOD_ORDER)
    tbl.columns = [f"k={c}" for c in tbl.columns]
    tbl.index.name = None
    tbl.columns.name = None
    return tbl

per_metric     = {(c, m): pivot_metric(summary, m, c)
                  for c in CLASSIFIERS_TO_RUN for m in REPORT_COLS}
per_metric_tex = {(c, m): pivot_metric(summary_tex, m, c)
                  for c in CLASSIFIERS_TO_RUN for m in REPORT_COLS}

for clf_name in CLASSIFIERS_TO_RUN:
    for m in REPORT_COLS:
        print(f"\n=== {clf_name} | {METRIC_LABELS[m]} (mean +/- std over {len(SEEDS)} seeds) " + "=" * 20)
        print(per_metric[(clf_name, m)].to_string())


=== rf | Acc (mean +/- std over 3 seeds) ====================
                             k=5             k=10             k=15             k=20             k=25             k=30
CGFS-weighted    0.9664 ± 0.0129  0.9803 ± 0.0022  0.9794 ± 0.0004  0.9798 ± 0.0004  0.9982 ± 0.0002  0.9982 ± 0.0002
CGFS-unweighted  0.9626 ± 0.0140  0.9808 ± 0.0011  0.9917 ± 0.0082  0.9977 ± 0.0008  0.9983 ± 0.0002  0.9983 ± 0.0003
Random           0.9493 ± 0.0235  0.9700 ± 0.0130  0.9858 ± 0.0101  0.9933 ± 0.0048  0.9933 ± 0.0049  0.9952 ± 0.0014
RFE              0.9716 ± 0.0002  0.9941 ± 0.0047  0.9969 ± 0.0005  0.9973 ± 0.0009  0.9978 ± 0.0012  0.9982 ± 0.0003
XGBoost          0.9829 ± 0.0030  0.9964 ± 0.0007  0.9971 ± 0.0006  0.9971 ± 0.0005  0.9971 ± 0.0004  0.9973 ± 0.0004
MutualInfo       0.9558 ± 0.0008  0.9717 ± 0.0009  0.9804 ± 0.0012  0.9797 ± 0.0008  0.9789 ± 0.0007  0.9792 ± 0.0004
ReliefF          0.9478 ± 0.0320  0.9676 ± 0.0111  0.9970 ± 0.0003  0.9976 ± 0.0004  0.9980 ± 0.0004  0.9982 ± 

In [17]:
# Delta against the random baseline, in units of the baseline's own spread.
for clf_name in CLASSIFIERS_TO_RUN:
    sub = raw[raw.classifier == clf_name]
    rand = sub[sub.method == "Random"].groupby("k")["macro_f1"].agg(["mean", "std"])
    a = (sub.groupby(["method", "k"])["macro_f1"].agg(["mean", "std"]).reset_index())
    print(f"\n### {SPEC.name} / {clf_name} -- macro-F1 vs random")
    print(f"{'method':17s} {'k':>4s} {'macro-F1':>19s} {'dF1 vs rand':>12s} {'z':>7s}")
    for _, r in a.sort_values(["k", "method"]).iterrows():
        if r["method"] == "Random":
            delta = z = ""
        else:
            rm, rs = rand.loc[r["k"], "mean"], rand.loc[r["k"], "std"]
            dv = r["mean"] - rm
            delta = f"{dv:+.4f}"
            z = f"{dv / rs:+.2f}" if rs and rs > 1e-9 else "n/a"
        sd = 0.0 if pd.isna(r["std"]) else r["std"]
        print(f"{r['method']:17s} {int(r['k']):4d} {r['mean']:.4f}+-{sd:.4f}"
              f"{delta:>15s} {z:>7s}")


### RT-IoT2022 / rf -- macro-F1 vs random
method               k            macro-F1  dF1 vs rand       z
ANOVA                5 0.8657+-0.0154        +0.2254   +2.07
CGFS-unweighted      5 0.7345+-0.0931        +0.0942   +0.87
CGFS-weighted        5 0.8365+-0.0313        +0.1962   +1.80
Chi2                 5 0.8144+-0.0109        +0.1741   +1.60
Correlation          5 0.7514+-0.0109        +0.1111   +1.02
L1                   5 0.7563+-0.0097        +0.1160   +1.07
MutualInfo           5 0.6287+-0.0136        -0.0116   -0.11
RFE                  5 0.7943+-0.0079        +0.1540   +1.41
Random               5 0.6403+-0.1089                       
ReliefF              5 0.6384+-0.0921        -0.0018   -0.02
XGBoost              5 0.7659+-0.0412        +0.1256   +1.15
mRMR                 5 0.9502+-0.0141        +0.3099   +2.85
ANOVA               10 0.8781+-0.0145        +0.1182   +1.00
CGFS-unweighted     10 0.8622+-0.0130        +0.1023   +0.86
CGFS-weighted       10 0.8700+-0.0207  

## 15. Selection stability and weighted-vs-unweighted overlap

Mean pairwise Jaccard overlap between the feature sets a selector picks on
different seeds. Since the seeds differ only in the train/test split, this is
selection stability under resampling: 1.000 means the selector returns the same
subset whichever 70% of the data it sees.

The second block is the one that matters for the paper: how much the weighting
actually moves the selected set relative to the unweighted greedy, and relative
to plain MI ranking.

In [18]:
from itertools import combinations

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 1.0

clf0 = CLASSIFIERS_TO_RUN[0]
base = raw[raw.classifier == clf0]

rows = []
for name in METHOD_ORDER:
    for k in sorted(base["k"].unique()):
        sets = [r.split("|") for r in
                base[(base.method == name) & (base.k == k)].sort_values("seed")["features"]]
        pairs = [jaccard(a, b) for a, b in combinations(sets, 2)]
        rows.append({"method": name, "k": k, "jaccard": float(np.mean(pairs)) if pairs else 1.0})

stability = (pd.DataFrame(rows).pivot(index="method", columns="k", values="jaccard")
             .reindex(METHOD_ORDER).round(3))
stability.columns = [f"k={c}" for c in stability.columns]
stability.index.name = None
print("Mean pairwise Jaccard overlap of selected feature sets across seeds")
print("(1.000 = the same subset is chosen regardless of which split is drawn)\n")
print(stability.to_string())

Mean pairwise Jaccard overlap of selected feature sets across seeds
(1.000 = the same subset is chosen regardless of which split is drawn)

                   k=5   k=10   k=15   k=20   k=25   k=30
CGFS-weighted    0.389  0.495  0.625  0.661  0.654  0.706
CGFS-unweighted  0.667  0.632  0.611  0.583  0.708  0.732
Random           0.037  0.076  0.086  0.166  0.212  0.233
RFE              0.587  0.632  0.801  0.765  0.787  0.748
XGBoost          0.587  0.768  0.917  0.794  0.686  0.683
MutualInfo       1.000  1.000  0.917  0.937  1.000  1.000
ReliefF          0.587  0.818  0.917  0.905  0.876  0.915
L1               0.778  0.692  0.778  0.733  0.699  0.801
Correlation      1.000  0.879  1.000  0.937  0.949  0.957
ANOVA            0.778  1.000  1.000  0.937  1.000  0.957
Chi2             1.000  1.000  1.000  1.000  0.949  0.957
mRMR             1.000  1.000  1.000  1.000  1.000  0.917


In [19]:
def sets_for(name, k, seed):
    r = base[(base.method == name) & (base.k == k) & (base.seed == seed)]["features"]
    return set(r.iloc[0].split("|")) if len(r) else set()

pairs = [("CGFS-weighted", "CGFS-unweighted"), ("CGFS-weighted", "MutualInfo"),
         ("CGFS-unweighted", "MutualInfo")]
if INCLUDE_MRMR:
    pairs.append(("CGFS-weighted", "mRMR"))

rows = []
for a, b in pairs:
    row = {"pair": f"{a} vs {b}"}
    for k in sorted(base["k"].unique()):
        vals = [jaccard(sets_for(a, k, s), sets_for(b, k, s)) for s in SEEDS]
        row[f"k={k}"] = round(float(np.mean(vals)), 3)
    rows.append(row)
print("Mean Jaccard overlap between selectors (averaged over seeds)\n")
print(pd.DataFrame(rows).set_index("pair").to_string())

Mean Jaccard overlap between selectors (averaged over seeds)

                                    k=5   k=10   k=15   k=20   k=25   k=30
pair                                                                      
CGFS-weighted vs CGFS-unweighted  0.217  0.278  0.186  0.365  0.403  0.477
CGFS-weighted vs MutualInfo       0.000  0.111  0.306  0.350  0.431  0.476
CGFS-unweighted vs MutualInfo     0.000  0.072  0.141  0.213  0.230  0.268
CGFS-weighted vs mRMR             0.074  0.133  0.250  0.319  0.365  0.429


In [20]:
# What the weighting adds and drops at k=10, per seed.
K_SHOW = 10 if 10 in list(base["k"].unique()) else int(base["k"].min())
for s in SEEDS:
    w_set = sets_for("CGFS-weighted", K_SHOW, s)
    u_set = sets_for("CGFS-unweighted", K_SHOW, s)
    print(f"\nseed {s}, k={K_SHOW}")
    print(f"  shared        ({len(w_set & u_set)}): {sorted(w_set & u_set)}")
    print(f"  weighted only ({len(w_set - u_set)}): {sorted(w_set - u_set)}")
    print(f"  unweighted only ({len(u_set - w_set)}): {sorted(u_set - w_set)}")


seed 0, k=10
  shared        (4): ['bwd_data_pkts_tot', 'bwd_header_size_min', 'flow_iat.tot', 'fwd_header_size_tot']
  weighted only (6): ['active.avg', 'bwd_pkts_per_sec', 'flow_iat.avg', 'flow_pkts_payload.std', 'fwd_init_window_size', 'fwd_pkts_payload.avg']
  unweighted only (6): ['active.max', 'bwd_iat.avg', 'flow_iat.std', 'flow_pkts_payload.max', 'payload_bytes_per_second', 'service']

seed 1, k=10
  shared        (5): ['flow_iat.tot', 'flow_pkts_payload.max', 'fwd_header_size_tot', 'payload_bytes_per_second', 'service']
  weighted only (5): ['active.avg', 'bwd_pkts_tot', 'flow_iat.avg', 'fwd_header_size_max', 'fwd_pkts_payload.avg']
  unweighted only (5): ['active.max', 'bwd_header_size_min', 'bwd_iat.avg', 'bwd_pkts_payload.tot', 'flow_iat.std']

seed 2, k=10
  shared        (4): ['active.avg', 'bwd_header_size_min', 'flow_iat.tot', 'fwd_header_size_tot']
  weighted only (6): ['bwd_data_pkts_tot', 'flow_iat.avg', 'flow_pkts_payload.std', 'flow_pkts_per_sec', 'fwd_init_window

## 16. Coverage-error traces

`E_i = d − Σ w_i g_i` after each greedy round, for both variants. Non-increasing
by construction; the gap between the two curves is what the relevance weighting
buys in coverage terms.

In [21]:
ctx0 = build_context(SEEDS[0])
_, err_w = cgfs(ctx0.sim, ctx0.w, KMAX)
_, err_u = cgfs(ctx0.sim, np.ones(ctx0.d), KMAX)

trace = pd.DataFrame({
    "i": range(1, len(err_w) + 1),
    "E_i weighted": np.round(err_w, 4),
    "E_i unweighted": np.round(err_u, 4),
}).set_index("i")
print(f"d = {ctx0.d}, seed {SEEDS[0]}\n")
print(trace.to_string())

d = 77, seed 0

    E_i weighted  E_i unweighted
i                               
1        53.5669         54.8009
2        41.5462         43.6656
3        36.1771         38.0542
4        31.9738         33.7786
5        27.9986         30.3588
6        24.7677         27.3071
7        21.5865         24.3223
8        18.9172         21.4386
9        17.1120         18.7648
10       15.6665         17.1941
11       14.4216         15.6758
12       13.2829         14.4415
13       12.2129         13.3260
14       11.2186         12.5992
15       10.2884         11.9087
16        9.3882         11.2310
17        8.5478         10.5833
18        7.8731          9.9425
19        7.2017          9.3053
20        6.6066          8.6757
21        6.1076          8.0688
22        5.6297          7.4719
23        5.2427          6.9098
24        4.8766          6.5104
25        4.5204          6.1155
26        4.1920          5.7437
27        3.8735          5.3801
28        3.5609          5

## 17. Timing

* **prep** — building `sim` and the plug-in MI, once per seed. Timed separately
  because different selectors depend on different halves of it.
* **select** — the selector's own call at that budget. Classifier-independent,
  so it is de-duplicated below.
* **select + prep** — the same, plus the shared prep that selector consumes
  (§10). This is the column to quote when comparing methods: `MutualInfo` reads a
  precomputed score and so measures ~0 on its own, which says nothing about what
  it costs to produce.
* **fit / infer** — training on the selected subset and predicting the whole
  test split.

In [22]:
# --- selection time: classifier-independent, so collapse it -----------------
uniq = raw.drop_duplicates(["seed", "method", "k"])

for col in ("select_time_s", "select_total_s"):
    t = uniq.groupby(["method", "k"])[col].agg(["mean", "std"]).reset_index()
    t["cell"] = [fmt(mu, sd, 3) for mu, sd in zip(t["mean"], t["std"])]
    tbl = t.pivot(index="method", columns="k", values="cell").reindex(METHOD_ORDER)
    tbl.columns = [f"k={c}" for c in tbl.columns]
    tbl.index.name = None; tbl.columns.name = None
    mode = "timed at each budget" if SELECT_PER_K else f"one run at k={KMAX}, reused"
    print(f"\n{METRIC_LABELS[col]} in seconds, mean +/- std over {len(SEEDS)} seeds "
          f"(SELECT_PER_K={SELECT_PER_K}: {mode})\n")
    print(tbl.to_string())

# --- the shared prep, and who pays for it -----------------------------------
p = raw.drop_duplicates("seed")
print(f"\nShared prep per seed: "
      f"sim {fmt(p['prep_sim_s'].mean(), p['prep_sim_s'].std(), 3)} s | "
      f"MI {fmt(p['prep_mi_s'].mean(), p['prep_mi_s'].std(), 3)} s")
print("charged to:", {k: '+'.join(v) for k, v in SELECTOR_PREP.items()} or "nothing")

gap = (uniq.groupby("method")[["select_time_s", "select_total_s"]].mean()
           .reindex(METHOD_ORDER).round(3))
gap["prep_charged"] = (gap["select_total_s"] - gap["select_time_s"]).round(3)
gap.index.name = None
print("\nMean over budgets and seeds:\n")
print(gap.to_string())


Select(s) in seconds, mean +/- std over 3 seeds (SELECT_PER_K=True: timed at each budget)

                            k=5            k=10            k=15            k=20            k=25            k=30
CGFS-weighted     0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000
CGFS-unweighted   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000   0.001 ± 0.000
Random            0.000 ± 0.000   0.000 ± 0.000   0.000 ± 0.000   0.000 ± 0.000   0.000 ± 0.000   0.000 ± 0.000
RFE              59.439 ± 0.943  56.643 ± 0.343  53.553 ± 0.872  50.726 ± 0.837  47.977 ± 1.232  44.098 ± 0.917
XGBoost          15.175 ± 0.412  15.631 ± 0.245  15.044 ± 0.374  15.714 ± 0.541  14.887 ± 0.140  15.085 ± 0.341
MutualInfo       19.125 ± 0.085  19.056 ± 0.192  18.980 ± 0.048  18.966 ± 0.022  19.005 ± 0.073  18.960 ± 0.053
ReliefF           0.758 ± 0.022   0.757 ± 0.028   0.762 ± 0.033   0.755 ± 0.028   0.763 ± 0.029   0.756 ± 0.022
L1          

## 18. Save results

In [23]:
os.makedirs(OUT_DIR, exist_ok=True)
tag = DATASET_KEY
paths = {
    "raw": os.path.join(OUT_DIR, f"cgfs_{tag}_raw.csv"),
    "agg": os.path.join(OUT_DIR, f"cgfs_{tag}_agg.csv"),
    "full": os.path.join(OUT_DIR, f"cgfs_{tag}_full_ceiling.csv"),
}
raw.to_csv(paths["raw"], index=False)
agg.to_csv(paths["agg"], index=False)
full.to_csv(paths["full"], index=False)
for v in paths.values():
    print("wrote", v)

wrote ./cgfs_rt-iot_raw.csv
wrote ./cgfs_rt-iot_agg.csv
wrote ./cgfs_rt-iot_full_ceiling.csv


In [24]:
def latex_table(metric, clf_name):
    # escape=False because the cells carry a deliberate $\pm$; the only other
    # content is method names and numbers, which need no escaping.
    tbl = per_metric_tex[(clf_name, metric)].copy()
    label = METRIC_LABELS[metric].replace("(w)", " (weighted)")
    body = tbl.to_latex(escape=False, column_format="l" + "c" * tbl.shape[1])
    caption = (f"{label} on {SPEC.name} ({clf_name}), mean $\\pm$ standard deviation "
               f"over {len(SEEDS)} seeds, each an independent stratified "
               f"70/30 split.")
    return ("\\begin{table}[t]\n\\centering\n\\small\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{tab:cgfs-{tag}-{clf_name}-{metric.replace('_', '-')}}}\n"
            f"{body}"
            "\\end{table}\n")

tex_path = os.path.join(OUT_DIR, f"cgfs_{tag}_tables.tex")
with open(tex_path, "w") as fh:
    fh.write("% Generated by the CGFS weighted/unweighted benchmark notebook.\n")
    fh.write("% Requires \\usepackage{booktabs} in the preamble.\n\n")
    for clf_name in CLASSIFIERS_TO_RUN:
        for m in REPORT_COLS:
            fh.write(latex_table(m, clf_name))
            fh.write("\n")
print("wrote", tex_path)
print(latex_table("fpr", CLASSIFIERS_TO_RUN[0]))

wrote ./cgfs_rt-iot_tables.tex
\begin{table}[t]
\centering
\small
\caption{FPR on RT-IoT2022 (rf), mean $\pm$ standard deviation over 3 seeds, each an independent stratified 70/30 split.}
\label{tab:cgfs-rt-iot-rf-fpr}
\begin{tabular}{lcccccc}
\toprule
 & k=5 & k=10 & k=15 & k=20 & k=25 & k=30 \\
\midrule
CGFS-weighted & 0.0029 $\pm$ 0.0011 & 0.0017 $\pm$ 0.0002 & 0.0018 $\pm$ 0.0000 & 0.0017 $\pm$ 0.0000 & 0.0002 $\pm$ 0.0000 & 0.0002 $\pm$ 0.0000 \\
CGFS-unweighted & 0.0092 $\pm$ 0.0064 & 0.0016 $\pm$ 0.0001 & 0.0007 $\pm$ 0.0007 & 0.0002 $\pm$ 0.0001 & 0.0001 $\pm$ 0.0000 & 0.0001 $\pm$ 0.0000 \\
Random & 0.0131 $\pm$ 0.0098 & 0.0051 $\pm$ 0.0054 & 0.0012 $\pm$ 0.0009 & 0.0006 $\pm$ 0.0004 & 0.0006 $\pm$ 0.0004 & 0.0004 $\pm$ 0.0001 \\
RFE & 0.0024 $\pm$ 0.0000 & 0.0005 $\pm$ 0.0004 & 0.0003 $\pm$ 0.0000 & 0.0002 $\pm$ 0.0001 & 0.0002 $\pm$ 0.0001 & 0.0002 $\pm$ 0.0000 \\
XGBoost & 0.0015 $\pm$ 0.0003 & 0.0003 $\pm$ 0.0001 & 0.0003 $\pm$ 0.0000 & 0.0003 $\pm$ 0.0000 & 0.0003 $\pm$ 0

## 19. Optional — per-class report for one configuration

For the per-class F1 / recall tables: pick a method, a budget and a seed.

In [25]:
INSPECT_METHOD = "CGFS-weighted"
INSPECT_K      = 10
INSPECT_SEED   = SEEDS[0]
INSPECT_CLF    = CLASSIFIERS_TO_RUN[0]

ctx = build_context(INSPECT_SEED)
idx = _take(SELECTORS[INSPECT_METHOD](ctx, INSPECT_K), INSPECT_K, INSPECT_METHOD)

clf = CLASSIFIERS[INSPECT_CLF](ctx.seed)
clf.fit(ctx.Ztr[:, idx], ctx.ytr)
pred = clf.predict(ctx.Zte[:, idx])

fpr_macro, fpr_w = fpr_from_confusion(ctx.yte, pred)
print(f"{INSPECT_METHOD}, k={INSPECT_K}, seed={INSPECT_SEED}, clf={INSPECT_CLF}")
print("features:", [ctx.feat_names[i] for i in idx])
print("weights :", [round(float(ctx.w[i]), 4) for i in idx])
print(f"FPR macro={fpr_macro:.4f}  weighted={fpr_w:.4f}")
print()
print(classification_report(ctx.yte, pred,
                            target_names=[str(c) for c in LE_Y.classes_],
                            zero_division=0, digits=4))

CGFS-weighted, k=10, seed=0, clf=rf
features: ['fwd_init_window_size', 'fwd_header_size_tot', 'flow_pkts_payload.std', 'flow_iat.avg', 'bwd_pkts_per_sec', 'flow_iat.tot', 'active.avg', 'bwd_data_pkts_tot', 'bwd_header_size_min', 'fwd_pkts_payload.avg']
weights : [1.7526, 1.5037, 1.2159, 1.4597, 1.3312, 1.4582, 1.4752, 0.9568, 1.1379, 1.7726]
FPR macro=0.0019  weighted=0.0006

                            precision    recall  f1-score   support

            ARP_poisioning     0.9532    0.9548    0.9540      2325
            DDOS_Slowloris     0.9108    0.8938    0.9022       160
             DOS_SYN_Hping     1.0000    1.0000    1.0000     28398
              MQTT_Publish     0.9976    0.9976    0.9976      1244
Metasploit_Brute_Force_SSH     0.7000    0.6364    0.6667        11
             NMAP_FIN_SCAN     0.6154    1.0000    0.7619         8
         NMAP_OS_DETECTION     0.5721    0.3833    0.4591       600
             NMAP_TCP_scan     1.0000    0.9934    0.9967       301
        